# 🎬 Modern Sentiment Analysis on IMDB Movie Reviews using Hugging Face Transformers

This notebook demonstrates a **modern, industry-standard sentiment classification pipeline** using the Hugging Face **Transformers** library.

### Why Hugging Face Transformers?
* **Robust & Portable:** Works natively with PyTorch or TensorFlow, utilizing fast Rust-compiled tokenizers that install seamlessly on any platform (including macOS Apple Silicon and Python 3.12).
* **Zero Outdated Libraries:** Avoids deprecated package structures and avoids platform-specific preprocessing libraries like `tensorflow-text` entirely.
* **Production-Ready:** Hugging Face is the modern industry standard for pre-trained language and vision models.

## 1. Imports and Environment Check

In [1]:
import os
import torch
import transformers
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print(f"Transformers Version: {transformers.__version__}")
print(f"PyTorch Version: {torch.__version__}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Apple Silicon (MPS) Available: {torch.backends.mps.is_available()}")

/Users/adithya/Development/adithyasean/computer-vision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers Version: 5.9.0
PyTorch Version: 2.12.0
TensorFlow Version: 2.21.0
Apple Silicon (MPS) Available: True


## 2. Instant Out-of-the-Box Inference (Zero-Shot)

Using a pre-trained model (like `distilbert-base-uncased-finetuned-sst-2-english`) through Hugging Face `pipeline` gives us instant access to a highly optimized, state-of-the-art sentiment classifier without needing any training at all. It handles downloading, tokenizing, and model inference automatically.

In [2]:
# Load the out-of-the-box sentiment analysis pipeline
print("Loading state-of-the-art sentiment analysis pipeline...")
nlp_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Test on custom movie reviews!
sample_reviews = [
    "The cinematography was absolutely breathtaking, and the acting was top-tier. A masterpiece!",
    "A total disaster. The plot makes no sense, and the acting is incredibly stiff. Do not waste your time.",
    "It had some decent moments and nice music, but the second half was slow and boring."
]

results = nlp_pipeline(sample_reviews)

for review, result in zip(sample_reviews, results):
    print(f"Review: \"{review}\"")
    print(f"Predicted Sentiment: {result['label']} (with {result['score'] * 100:.2f}% confidence)\n")

Loading state-of-the-art sentiment analysis pipeline...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 23055.37it/s]


Review: "The cinematography was absolutely breathtaking, and the acting was top-tier. A masterpiece!"
Predicted Sentiment: POSITIVE (with 99.99% confidence)

Review: "A total disaster. The plot makes no sense, and the acting is incredibly stiff. Do not waste your time."
Predicted Sentiment: NEGATIVE (with 99.97% confidence)

Review: "It had some decent moments and nice music, but the second half was slow and boring."
Predicted Sentiment: NEGATIVE (with 99.78% confidence)



## 3. Custom Model Fine-Tuning

If you want to train a custom model on your own dataset, Hugging Face provides the extremely elegant `Trainer` API.

First, we load a small subset of the IMDB reviews using TensorFlow Datasets.

In [3]:
print("Loading subset of IMDB dataset...")
# Load small subsets for lightning-fast training
ds_train, ds_test = tfds.load(
    "imdb_reviews",
    split=["train[:200]", "test[:50]"],
    as_supervised=True,
    batch_size=-1
)

train_texts = [text.decode("utf-8") for text in ds_train[0].numpy()]
train_labels = ds_train[1].numpy().tolist()

test_texts = [text.decode("utf-8") for text in ds_test[0].numpy()]
test_labels = ds_test[1].numpy().tolist()

print(f"Loaded {len(train_texts)} training examples and {len(test_texts)} validation examples.")

Loading subset of IMDB dataset...
Loaded 200 training examples and 50 validation examples.


I0000 00:00:1780341523.215851 11086468 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


### Tokenize the Dataset
We load a modern pre-trained WordPiece tokenizer corresponding to a lightweight `bert-tiny` model.

In [4]:
model_name = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize text and pad/truncate to 128 tokens
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# Simple PyTorch Dataset wrapper
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(train_encodings, train_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)
print("Datasets ready!")

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

### Load the BERT Tiny Model & Fine-Tune
We load a pre-trained sequence classifier and train it using the standard `Trainer` API.

In [ ]:
# Load BERT Tiny for sequence classification with 2 output labels (positive/negative)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Setup training configurations
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Fine-tuning modern BERT classifier...")
trainer.train()
print("Training complete!")